<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-11/AI_Modul_11.6_Praktikum_Face_Detection.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 11.6: Praktikum Face Detection
**Mata Kuliah:** Visi Komputer dan Pengolahan Citra Digital  
**Institut Pertanian Stiper (INSTIPER) Yogyakarta**  

---

### Capaian Pembelajaran Praktikum:
1. Memuat dan mengonfigurasi model detektor `cv2.CascadeClassifier` dari berkas XML bawaan OpenCV.
2. Menghitung dan membuktikan efisiensi evaluasi kotak citra integral $O(1)$ secara matematis.
3. Melakukan deteksi wajah pekerja kebun multi-skala dengan `detectMultiScale`.
4. Menganalisis trade-off parameter `scaleFactor` dan `minNeighbors` pada eliminasi alarm palsu.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f'OpenCV Version: {cv2.__version__}')

# Verifikasi ketersediaan berkas Haar Cascade frontal face
xml_file = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
print(f'Path Model Haar Cascade: {xml_file}')
print(f'Eksistensi Berkas XML   : {os.path.exists(xml_file)}')


## 1. Pembuktian Evaluasi Kotak Cepat pada Citra Integral O(1)
Mendemonstrasikan pembuatan citra integral dan penghitungan jumlah intensitas sembarang kotak hanya dengan 4 akses memori.

In [ ]:
# Matriks citra kecil 4x4
sample_mat = np.array([
    [2, 3, 1, 4],
    [4, 1, 5, 2],
    [3, 2, 2, 1],
    [1, 4, 3, 2]
], dtype=np.float32)

# Hitung citra integral OpenCV
integral_img = cv2.integral(sample_mat)

# Evaluasi kotak x: 1..2, y: 1..2 (nilai asli: 1 + 5 + 2 + 2 = 10)
# Perhatikan bahwa cv2.integral menambahkan padding 0 pada baris dan kolom 0 (dimensi H+1, W+1)
x1, y1 = 1, 1
x2, y2 = 3, 3 # batas inklusif di integral index

sum_integral = integral_img[y2, x2] + integral_img[y1, x1] - integral_img[y1, x2] - integral_img[y2, x1]
sum_manual = np.sum(sample_mat[1:3, 1:3])

print('Matriks Citra Asli (4x4):\n', sample_mat)
print('Matriks Citra Integral (5x5):\n', integral_img)
print(f'Jumlah Intensitas via Citra Integral : {sum_integral:.0f}')
print(f'Jumlah Intensitas via Penjumlahan Langsung: {sum_manual:.0f}')
assert sum_integral == sum_manual, 'Kalkulasi citra integral tidak cocok!'


## 2. Pembangkitan Citra Sintetis Pekerja Kebun dan Deteksi Wajah Multi-Skala
Membangkitkan citra simulasi pekerja kebun kelapa sawit dan menjalankan `detectMultiScale`.

In [ ]:
detector = cv2.CascadeClassifier(xml_file)

# Buat kanvas pekerja
H, W = 400, 600
canvas = np.full((H, W, 3), 170, dtype=np.uint8)

# Gambar wajah pekerja mandor
cv2.ellipse(canvas, (250, 200), (55, 75), 0, 0, 360, (185, 155, 135), -1)
cv2.circle(canvas, (230, 180), 7, (30, 20, 15), -1)
cv2.circle(canvas, (270, 180), 7, (30, 20, 15), -1)
cv2.line(canvas, (220, 168), (240, 168), (20, 20, 20), 3)
cv2.line(canvas, (260, 168), (280, 168), (20, 20, 20), 3)
cv2.ellipse(canvas, (250, 235), (20, 8), 0, 0, 180, (40, 30, 100), -1)

gray_canvas = cv2.cvtColor(canvas, cv2.COLOR_BGR2GRAY)

# Deteksi MultiScale
faces = detector.detectMultiScale(gray_canvas, scaleFactor=1.1, minNeighbors=3, minSize=(40, 40))
print(f'Jumlah Wajah Terdeteksi pada Citra: {len(faces)}')

annotated = canvas.copy()
for (x, y, w, h) in faces:
    cv2.rectangle(annotated, (x, y), (x+w, y+h), (0, 255, 0), 2)
    cv2.putText(annotated, 'Mandor Kebun', (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2)


## 3. Visualisasi Hasil Deteksi Wajah Haar Cascade
Menampilkan citra grayscale dan citra anotasi bounding box hasil inferensi kaskade.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.imshow(gray_canvas, cmap='gray')
ax1.set_title('1. Citra Grayscale Masukan Detektor', fontsize=11, fontweight='bold')
ax1.axis('off')

ax2.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
ax2.set_title(f'2. Hasil Deteksi Wajah ({len(faces)} Terdeteksi)', fontsize=11, fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.savefig('praktikum_deteksi_wajah_11_6.png', dpi=150)
plt.show()
